# Gold Claims Analytics

## Purpose

I use this notebook as one of the transformation sources for the Gold Lakeflow
pipeline.

I create business-facing analytical datasets from the Gold claim fact model.

The notebook defines:

- `monthly_claim_kpis`
- `fraud_summary`
- `provider_performance`

### Primary source

`health_insurance.gold.fact_claim`

### Additional source

`health_insurance.gold.dim_provider`

### Analytical approach

I use the claim-level fact table as the common source for reusable measures such
as claim volume, financial amounts, fraud incidence, late submissions, and
utilization.

I keep these outputs as separate materialized views because each dataset has a
different analytical grain and business purpose.

Lakeflow can determine their dependencies from the datasets they read and
refresh the analytical outputs after their upstream Gold models are available.

In [0]:
# importing the Lakeflow API and Spark functions used by the analytical models.

from pyspark import pipelines as dp
from pyspark.sql import functions as F

CATALOG = "health_insurance"

# referencing datasets managed within the same Gold pipeline.

FACT_SOURCE = "fact_claim"
PROVIDER_SOURCE = "dim_provider"

## Monthly claim KPIs

I create a monthly analytical dataset for monitoring claim activity and
financial performance over time.

### Grain

One row represents one claim year-month.

### Metrics

I calculate:

- total claim volume
- total claim amount
- average claim amount
- total patient out-of-pocket amount
- estimated insurer amount
- fraudulent claim count
- fraud rate
- late claim count
- late-submission rate
- average claim submission delay

This dataset is designed for trend reporting and dashboarding.

In [0]:
# defining the pipeline-managed monthly Claims KPI dataset.

@dp.materialized_view(
    name="monthly_claim_kpis",
    comment="Monthly claims volume, financial, fraud, and submission KPIs."
)
def monthly_claim_kpis():

    fact_df = spark.read.table(
        FACT_SOURCE
    )

    monthly_df = (
        fact_df

        .groupBy(
            "claim_year",
            "claim_month",
            "claim_year_month"
        )

        .agg(
            F.count("*").alias(
                "total_claims"
            ),

            F.sum("claim_amount").alias(
                "total_claim_amount"
            ),

            F.avg("claim_amount").alias(
                "average_claim_amount"
            ),

            F.sum(
                "patient_out_of_pocket_amount"
            ).alias(
                "total_patient_out_of_pocket_amount"
            ),

            F.sum(
                "estimated_insurer_amount"
            ).alias(
                "total_estimated_insurer_amount"
            ),

            # I am counting only Claims with a reliable fraud label.
            F.count(
                "is_fraudulent"
            ).alias(
                "fraud_labeled_claims"
            ),

            # I am exposing ambiguous source fraud labels as a quality metric.
            F.sum(
                F.when(
                    F.col("fraud_label_conflict") == True,
                    1
                ).otherwise(0)
            ).alias(
                "fraud_label_conflicts"
            ),

            F.sum(
                F.when(
                    F.col("is_fraudulent") == True,
                    1
                ).otherwise(0)
            ).alias(
                "fraudulent_claims"
            ),

            F.sum(
                F.when(
                    F.col("claim_submitted_late") == True,
                    1
                ).otherwise(0)
            ).alias(
                "late_claims"
            ),

            F.avg(
                "claim_submission_delay_days"
            ).alias(
                "average_submission_delay_days"
            )
        )
    )

    return (
        monthly_df

        # I am calculating fraud rate only from Claims
        # whose historical fraud label is reliable.
        .withColumn(
            "fraud_rate_pct",
            F.when(
                F.col("fraud_labeled_claims") > 0,
                F.round(
                    (
                        F.col("fraudulent_claims")
                        / F.col("fraud_labeled_claims")
                    ) * 100,
                    2
                )
            )
        )

        .withColumn(
            "late_submission_rate_pct",
            F.when(
                F.col("total_claims") > 0,
                F.round(
                    (
                        F.col("late_claims")
                        / F.col("total_claims")
                    ) * 100,
                    2
                )
            )
        )

        .withColumn(
            "average_claim_amount",
            F.round(
                F.col("average_claim_amount"),
                2
            )
        )

        .withColumn(
            "average_submission_delay_days",
            F.round(
                F.col("average_submission_delay_days"),
                2
            )
        )

        .withColumn(
            "_gold_transformed_at",
            F.current_timestamp()
        )
    )

## Fraud summary

I create a fraud-oriented analytical dataset using the historical fraud label
provided by the claims source.

### Grain

One row represents one combination of:

- claim amount band
- service type

### Metrics

I calculate:

- total claims
- total claim amount
- fraudulent claims
- fraud rate
- fraudulent claim amount
- average fraudulent claim amount

I use `is_fraudulent` only as the historical outcome supplied by the source
dataset. This dataset summarizes fraud patterns; it does not perform fraud
prediction.

In [0]:
# I am defining the pipeline-managed historical fraud summary.

@dp.materialized_view(
    name="fraud_summary",
    comment="Historical fraud patterns by claim amount band and service type."
)
def fraud_summary():

    fact_df = spark.read.table(
        FACT_SOURCE
    )

    fraud_df = (
        fact_df

        .groupBy(
            "claim_amount_band",
            "service_type"
        )

        .agg(
            # I am keeping total Claim volume for full business context.
            F.count("*").alias(
                "total_claims"
            ),

            F.sum(
                "claim_amount"
            ).alias(
                "total_claim_amount"
            ),

            # I am counting only Claims whose fraud label is known.
            F.count(
                "is_fraudulent"
            ).alias(
                "fraud_labeled_claims"
            ),

            # I am exposing source fraud-label conflicts separately.
            F.sum(
                F.when(
                    F.col("fraud_label_conflict") == True,
                    1
                ).otherwise(0)
            ).alias(
                "fraud_label_conflicts"
            ),

            F.sum(
                F.when(
                    F.col("is_fraudulent") == True,
                    1
                ).otherwise(0)
            ).alias(
                "fraudulent_claims"
            ),

            # I am calculating the financial denominator only from
            # Claims with a reliable fraud classification.
            F.sum(
                F.when(
                    F.col("is_fraudulent").isNotNull(),
                    F.col("claim_amount")
                ).otherwise(
                    F.lit(0)
                )
            ).alias(
                "fraud_labeled_claim_amount"
            ),

            F.sum(
                F.when(
                    F.col("is_fraudulent") == True,
                    F.col("claim_amount")
                ).otherwise(
                    F.lit(0)
                )
            ).alias(
                "fraudulent_claim_amount"
            ),

            F.avg(
                F.when(
                    F.col("is_fraudulent") == True,
                    F.col("claim_amount")
                )
            ).alias(
                "average_fraudulent_claim_amount"
            )
        )
    )

    return (
        fraud_df

        # I am excluding ambiguous fraud labels from the rate denominator.
        .withColumn(
            "fraud_rate_pct",
            F.when(
                F.col("fraud_labeled_claims") > 0,
                F.round(
                    (
                        F.col("fraudulent_claims")
                        / F.col("fraud_labeled_claims")
                    ) * 100,
                    2
                )
            )
        )

        # I am comparing fraudulent Claim value only against
        # Claim value with a reliable fraud classification.
        .withColumn(
            "fraud_claim_amount_share_pct",
            F.when(
                F.col("fraud_labeled_claim_amount") > 0,
                F.round(
                    (
                        F.col("fraudulent_claim_amount")
                        / F.col("fraud_labeled_claim_amount")
                    ) * 100,
                    2
                )
            )
        )

        .withColumn(
            "average_fraudulent_claim_amount",
            F.round(
                F.col(
                    "average_fraudulent_claim_amount"
                ),
                2
            )
        )

        .withColumn(
            "_gold_transformed_at",
            F.current_timestamp()
        )
    )

## Provider performance

I create a provider-level analytical dataset by combining the claim fact with
the Provider dimension.

### Grain

One row represents one provider profile.

### Metrics

I calculate:

- total claims
- distinct patients
- total claim amount
- average claim amount
- estimated insurer amount
- fraudulent claims
- provider fraud rate
- late claims
- late-submission rate
- average claim submission delay
- average procedure count
- average length of stay

I join `dim_provider` so the analytical output contains readable provider
attributes in addition to the deterministic `provider_key`.

In [0]:
# I am defining the pipeline-managed Provider performance dataset.

@dp.materialized_view(
    name="provider_performance",
    comment="Provider-level claims, financial, operational, and fraud performance."
)
def provider_performance():

    fact_df = (
        spark.read.table(
            FACT_SOURCE
        )
        .alias("fact")
    )

    provider_df = (
        spark.read.table(
            PROVIDER_SOURCE
        )
        .select(
            "provider_key",
            "hospital_id",
            "provider_type",
            "provider_specialty",
            "provider_city",
            "provider_state"
        )
        .alias("provider")
    )

    # I am explicitly selecting columns after the join so Provider
    # attributes cannot become ambiguous with columns retained in the fact.
    provider_claims_df = (
        fact_df

        .join(
            provider_df,
            F.col("fact.provider_key")
            == F.col("provider.provider_key"),
            how="left"
        )

        .select(
            F.col("fact.provider_key").alias(
                "provider_key"
            ),

            F.col("provider.hospital_id").alias(
                "hospital_id"
            ),

            F.col("provider.provider_type").alias(
                "provider_type"
            ),

            F.col("provider.provider_specialty").alias(
                "provider_specialty"
            ),

            F.col("provider.provider_city").alias(
                "provider_city"
            ),

            F.col("provider.provider_state").alias(
                "provider_state"
            ),

            F.col("fact.patient_key").alias(
                "patient_key"
            ),

            F.col("fact.claim_amount").alias(
                "claim_amount"
            ),

            F.col("fact.estimated_insurer_amount").alias(
                "estimated_insurer_amount"
            ),

            F.col("fact.is_fraudulent").alias(
                "is_fraudulent"
            ),

            F.col("fact.fraud_label_conflict").alias(
                "fraud_label_conflict"
            ),

            F.col("fact.claim_submitted_late").alias(
                "claim_submitted_late"
            ),

            F.col("fact.claim_submission_delay_days").alias(
                "claim_submission_delay_days"
            ),

            F.col("fact.number_of_procedures").alias(
                "number_of_procedures"
            ),

            F.col("fact.length_of_stay_days").alias(
                "length_of_stay_days"
            )
        )
    )

    provider_metrics_df = (
        provider_claims_df

        .groupBy(
            "provider_key",
            "hospital_id",
            "provider_type",
            "provider_specialty",
            "provider_city",
            "provider_state"
        )

        .agg(
            F.count("*").alias(
                "total_claims"
            ),

            F.countDistinct(
                "patient_key"
            ).alias(
                "distinct_patients"
            ),

            F.sum(
                "claim_amount"
            ).alias(
                "total_claim_amount"
            ),

            F.avg(
                "claim_amount"
            ).alias(
                "average_claim_amount"
            ),

            F.sum(
                "estimated_insurer_amount"
            ).alias(
                "total_estimated_insurer_amount"
            ),

            # I am counting only Claims with a reliable fraud label.
            F.count(
                "is_fraudulent"
            ).alias(
                "fraud_labeled_claims"
            ),

            # I am retaining fraud-label conflicts as a Provider quality metric.
            F.sum(
                F.when(
                    F.col("fraud_label_conflict") == True,
                    1
                ).otherwise(0)
            ).alias(
                "fraud_label_conflicts"
            ),

            F.sum(
                F.when(
                    F.col("is_fraudulent") == True,
                    1
                ).otherwise(0)
            ).alias(
                "fraudulent_claims"
            ),

            F.sum(
                F.when(
                    F.col("claim_submitted_late") == True,
                    1
                ).otherwise(0)
            ).alias(
                "late_claims"
            ),

            F.avg(
                "claim_submission_delay_days"
            ).alias(
                "average_submission_delay_days"
            ),

            F.avg(
                "number_of_procedures"
            ).alias(
                "average_procedure_count"
            ),

            F.avg(
                "length_of_stay_days"
            ).alias(
                "average_length_of_stay_days"
            )
        )
    )

    return (
        provider_metrics_df

        # I am excluding ambiguous fraud labels from Provider fraud rates.
        .withColumn(
            "fraud_rate_pct",
            F.when(
                F.col("fraud_labeled_claims") > 0,
                F.round(
                    (
                        F.col("fraudulent_claims")
                        / F.col("fraud_labeled_claims")
                    ) * 100,
                    2
                )
            )
        )

        .withColumn(
            "late_submission_rate_pct",
            F.when(
                F.col("total_claims") > 0,
                F.round(
                    (
                        F.col("late_claims")
                        / F.col("total_claims")
                    ) * 100,
                    2
                )
            )
        )

        .withColumn(
            "average_claim_amount",
            F.round(
                F.col("average_claim_amount"),
                2
            )
        )

        .withColumn(
            "average_submission_delay_days",
            F.round(
                F.col(
                    "average_submission_delay_days"
                ),
                2
            )
        )

        .withColumn(
            "average_procedure_count",
            F.round(
                F.col("average_procedure_count"),
                2
            )
        )

        .withColumn(
            "average_length_of_stay_days",
            F.round(
                F.col(
                    "average_length_of_stay_days"
                ),
                2
            )
        )

        .withColumn(
            "_gold_transformed_at",
            F.current_timestamp()
        )
    )

## Gold analytical outputs

When this notebook is added to the Gold Lakeflow pipeline, it defines three
business-facing materialized views.

### `monthly_claim_kpis`

Grain:

One row per claim year-month.

Purpose:

Monitor claim volume, financial performance, fraud incidence, and submission
behavior over time.

### `fraud_summary`

Grain:

One row per claim amount band and service type.

Purpose:

Analyze historical fraud frequency and financial exposure across claim
categories.

### `provider_performance`

Grain:

One row per Provider dimension profile.

Purpose:

Compare provider claim volume, patient reach, financial activity, fraud
incidence, late submissions, and utilization measures.

### Pipeline dependencies

The resulting Gold dependency graph is:

`silver.claims`
↓
`dim_patient`
`dim_provider`
`fact_claim`
↓
`monthly_claim_kpis`
`fraud_summary`
`provider_performance`

`provider_performance` additionally joins `dim_provider` using `provider_key`.

I do not manually persist or execute any of these datasets in this notebook.
Lakeflow will manage the analytical materialized views when the Gold pipeline is
eventually deployed and run.